# Создание приложений для генерации изображений (OpenAI)

Возможности больших языковых моделей выходят за рамки создания текста. Вы также можете генерировать изображения по текстовым описаниям. Изображения как модальность полезны в медицине, архитектуре, туризме, разработке игр, маркетинге и других сферах. В этом уроке мы работаем с современными моделями **GPT Image** на платформе OpenAI.

## Цели обучения

К концу урока вы сможете:

- Объяснить, что такое генерация изображений и где она применяется.
- Понять семейство моделей `gpt-image` и чем они отличаются от устаревших моделей DALL·E.
- Создать приложение для генерации изображений и редактировать изображения.

## Что такое генерация изображений?

Модели генерации изображений создают изображения на основе текстового запроса. Современные модели, такие как `gpt-image`, изучают взаимосвязь между текстом и изображениями во время обучения, а затем итеративно превращают случайный шум в изображение, соответствующее вашему запросу.

Два известных семейства моделей изображений:

- **`gpt-image` (OpenAI)** — текущее поколение моделей, используемых в этом уроке. Поддерживает генерацию изображений из текста и редактирование изображений (закрашивание с маской).
- **Midjourney** — популярная сторонняя модель с собственным сервисом и рабочим процессом в Discord.

> Старые модели изображений OpenAI — **DALL·E 2** и **DALL·E 3** — устарели. DALL·E 3 больше недоступна для новых развертываний, а функции вроде `create_variation` были в DALL·E 2. Для новых приложений используйте модели `gpt-image`.

> **Важно:** модели `gpt-image` возвращают сгенерированное изображение в виде **base64** (`b64_json`), а не URL. Ваш код декодирует base64-строку в байты и сохраняет — URL для скачивания изображения отсутствует.


## Создание вашего первого приложения для генерации изображений

Итак, что нужно для создания приложения для генерации изображений? Вам понадобятся следующие библиотеки:

- **python-dotenv**, настоятельно рекомендуется использовать эту библиотеку, чтобы хранить ваши секреты в файле *.env*, отдельно от кода.
- **openai**, эта библиотека необходима для взаимодействия с API OpenAI.
- **pillow**, для работы с изображениями в Python.


1. Создайте файл *.env* со следующим содержимым:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Соберите перечисленные выше библиотеки в файл с названием *requirements.txt*, например так:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затем создайте виртуальное окружение и установите библиотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows используйте следующие команды, чтобы создать и активировать виртуальное окружение:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавьте следующий код в файл под названием *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Создайте объект OpenAI (считывает OPENAI_API_KEY из вашего .env)
    client = openai.OpenAI()


    try:
        # Создайте изображение, используя API для генерации изображений
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введите ваш текст запроса здесь
            size='1024x1024',
            n=1
        )
        # Установите директорию для сохранённого изображения
        image_dir = os.path.join(os.curdir, 'images')

        # Если директории не существует, создайте её
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Инициализируйте путь к изображению (обратите внимание, что тип файла должен быть png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Модели gpt-image возвращают изображение в формате base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Отобразите изображение в программе по умолчанию для просмотра изображений
        image = Image.open(image_path)
        image.show()

    # обработка исключений
    except openai.BadRequestError as err:
        print(err)

    ```

Давайте объясним этот код:

- Во-первых, мы импортируем необходимые библиотеки, включая библиотеку OpenAI, библиотеку dotenv, модуль base64 и библиотеку Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- После этого мы создаём клиент, который читает API-ключ из вашего файла ``.env``.

    ```python
    # Создать объект OpenAI
    client = openai.OpenAI()
    ```

- Затем мы генерируем изображение:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введите здесь текст вашего запроса
        size='1024x1024',
        n=1
    )
    ```

    Модели `gpt-image` возвращают изображение в виде строки **base64** в `data[0].b64_json`. Мы декодируем её в байты и записываем в файл — URL для загрузки нет.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Наконец, мы открываем изображение и используем стандартный просмотрщик изображений для его отображения:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Подробнее о генерации изображения

Давайте рассмотрим параметры `images.generate`:

- **model** — модель изображения, которую нужно использовать (например, `gpt-image-1`).
- **prompt** — текстовый запрос, используемый для генерации изображения. Здесь это "Кролик на лошади, держащий леденец, на туманном лугу, где растут нарциссы".
- **size** — размер генерируемого изображения (например, `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n** — количество генерируемых изображений. Здесь мы генерируем одно.

> Модели изображений не принимают параметр `temperature` — это управление генерацией текста. Чтобы получить разнообразие, вызовите API снова; чтобы уменьшить разнообразие, сделайте запрос более конкретным.

## Дополнительные возможности генерации изображений

Вы увидели, как создать изображение всего несколькими строками Python. Модели `gpt-image` также могут **редактировать** существующее изображение. Предоставив изображение, необязательную **маску** (которая отмечает область для изменения) и запрос, вы можете изменить часть изображения — например, добавить шляпу нашему кролику.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# изменения также возвращаются в формате base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Исходное изображение содержит только кролика; итоговое изображение — с шляпой на кролике.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
